### PSD Pipeline
#### Table of Contents:
* [1. Environment Setup](#1-environment-setup)
    * [1.1 Library Imports](#11-library-imports)
    * [1.2 Dataset loading/inspection](#12-dataset-loadinginspection)
* [2. Data Segmentation](#data-segmentation)
* [3. Feature Extraction and Class Distribution](#3-feature-extraction--class-distribution)
    * [3.1 PSD](#31-psd-feature-extraction)
    * [3.2 Class Distribution](#32-class-distribution)
* [4. Classification Schemes](#4-classification-schemes)
    * [4.1 Emotional vs. Neutral](#41-emotional-vs-neutral-remap--class-distribution)
    * [4.2 Positive vs. Negative](#42-positive-vs-negative-remap--class-distribution)
* [5. Model Training](#5-model-training)
    * [5.1 XGBoost](#51-xgboost)
        * [Emotional vs. Neutral](#511-emotional-vs-neutral)




### 1. Environment Setup

##### 1.1 Library Imports

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.io import loadmat
import pywt as pwt
from pywt import wavedec

import matplotlib.pyplot as plt
import seaborn as sns

import optuna
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import LeaveOneGroupOut, StratifiedKFold, StratifiedGroupKFold, cross_val_score, cross_validate
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, roc_auc_score, classification_report


/Users/connor/miniconda3/envs/ml/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


##### 1.2 Dataset loading/inspection

In [2]:
dataset_path = Path('DEED')
eeg_dataset = []
subject_ids = []

for file in dataset_path.iterdir():
    mat = loadmat(file)
    eeg = mat['Data']
    fname = file.stem  
    label_part = [part for part in fname.split("_") if part.startswith("E")][0]
    subject_part = [part for part in fname.split("_") if part.startswith("S")][0]
    label = int(label_part[1:])  
    subject_id = subject_part[1:-1]  # last two digits = subject number
    
    eeg_dataset.append((eeg, label))
    subject_ids.append(subject_id)

print(f"Loaded {len(eeg_dataset)} trials.")
print(f"Unique subjects: {len(set(subject_ids))}")
print(f"Subject IDs: {sorted(set(subject_ids))}")
print("Example shapes:", [(arr.shape, lbl) for arr, lbl in eeg_dataset[:3]])

print("\nSample filename → label mapping:")
for file, (_, label) in zip(dataset_path.iterdir(), eeg_dataset[:10]):
    print(f"  {file.stem} → E{label}")

Loaded 533 trials.
Unique subjects: 34
Subject IDs: ['002', '003', '004', '005', '007', '011', '012', '013', '014', '015', '016', '017', '018', '020', '021', '022', '023', '024', '025', '027', '028', '029', '030', '031', '032', '033', '034', '035', '036', '037', '038', '039', '040', '042']
Example shapes: [((6, 290000), 2), ((6, 36000), 2), ((6, 51000), 3)]

Sample filename → label mapping:
  G_S0321_M1_E2_R1_N2_raw_ref → E2
  G_S0213_M3_E2_R7_N2_raw_ref → E2
  G_S0393_M2_E3_R5_REM_raw_ref → E3
  G_S0243_M3_E2_R2_N2_raw_ref → E2
  G_S0311_M3_E5_R2_N2_raw_ref → E5
  G_S0031_M1_E3_R4_nan_raw_ref → E3
  G_S0043_M2_E2_R5_N2_raw_ref → E2
  G_S0072_M1_E0_R11_N1_raw_ref → E0
  G_S0242_M1_E3_R3_W_raw_ref → E3
  G_S0342_M2_E3_R3_N2_raw_ref → E3


### 2. Data Segmentation

Done into 20s windows.

In [3]:
def segmentation(eeg_dataset, subject_ids, window_sec, fs):
    window_size = int(window_sec * fs)
    X = []
    y = []
    groups = []

    for (eeg_array, label), sid in zip(eeg_dataset, subject_ids):
        n_samples = eeg_array.shape[1]
        start = 0
        while start + window_size <= n_samples:
            window = eeg_array[:, start:start + window_size]
            X.append(window)
            y.append(label)
            groups.append(sid)
            start += window_size  

    return X, y, groups

windows, window_labels, window_groups = segmentation(eeg_dataset, subject_ids, 20, 200)
print(f"Total 20 second windows: {len(windows)}")
print(f"Unique subjects: {len(set(window_groups))}")

Total 20 second windows: 7490
Unique subjects: 34


### 3. Feature Extraction & Class Distribution

PSD features are extracted from each 20s window using Welch's method. For each channel, log band power 
and relative band power are computed across the five standard frequency bands (delta, theta, alpha, beta, 
gamma), alongside time-domain mean and variance. Windows are then remapped into two binary classification schemes: Emotional vs. Neutral and Positive vs. Negative.

##### 3.1 PSD Feature Extraction

In [ ]:
CHANNELS = ['F3', 'F4', 'FT7', 'FT8', 'T7', 'T8']
ASYM_PAIRS = [(0, 1), (2, 3), (4, 5)]  # F3/F4, FT7/FT8, T7/T8

def extract_dwt_features(segmented_windows, labels):
    features = []
    for window in segmented_windows:
        channel_features = []
        channel_coeffs = []

        # Per-channel per-band stats
        for ch in range(window.shape[0]):
            coeffs = wavedec(window[ch], 'db4', level=5)
            channel_coeffs.append(coeffs)
            for coeff in coeffs:
                channel_features.extend([
                    np.mean(coeff),
                    np.std(coeff),
                    np.var(coeff),
                    np.sum(coeff**2),        # band energy
                    np.max(np.abs(coeff)),   # peak amplitude
                ])

        for left, right in ASYM_PAIRS:
            for level in range(6):  # 6 coefficient arrays at level=5
                left_energy = np.sum(channel_coeffs[left][level]**2)
                right_energy = np.sum(channel_coeffs[right][level]**2)
                asymmetry = (left_energy - right_energy) / (left_energy + right_energy + 1e-8)
                channel_features.append(asymmetry)

        features.append(channel_features)

    X = np.array(features)
    y = np.array(labels)
    return X, y


##### 3.2 Class Distribution

In [5]:
X, y = extract_dwt_features(windows, window_labels)
print(X.shape)
print(y.shape) 

print("\n=== Total Class Distribution===")
unique, counts = np.unique(y, return_counts=True)
for label, count in zip(unique, counts):
    print(f"E{label}: {count} windows ({count/len(y)*100:.1f}%)")

X shape: (7490, 198), y shape: (7490,)
(7490, 198)
(7490,)

=== Total Class Distribution===
E0: 1294 windows (17.3%)
E1: 305 windows (4.1%)
E2: 1069 windows (14.3%)
E3: 2855 windows (38.1%)
E4: 1698 windows (22.7%)
E5: 269 windows (3.6%)


### 4. Classification Schemes

##### 4.1 Emotional vs. Neutral (remap + class distribution)  
Emotional {E1, E2, E4, E5} vs. Neutral Dream {E3} 

In [6]:
def remap_emotional_neutral(y):
    y = np.array(y)
    keep_mask = np.isin(y, [1, 2, 3, 4, 5])  
    new_y = np.zeros(len(y), dtype=int)
    new_y[np.isin(y, [1, 2, 4, 5])] = 1  
    return new_y[keep_mask], keep_mask

y_en, mask_en = remap_emotional_neutral(y)
X_en = X[mask_en]
groups_en = np.array(window_groups)[mask_en]

print(f"\n=== Emotional vs. Neutral Class Distribution ===")
print(f"  Total: {len(y_en)}")
print(f"  Neutral (0): {np.sum(y_en == 0)} ({np.sum(y_en == 0)/len(y_en)*100:.1f}%)")
print(f"  Emotional (1): {np.sum(y_en == 1)} ({np.sum(y_en == 1)/len(y_en)*100:.1f}%)")


=== Emotional vs. Neutral Class Distribution ===
  Total: 6196
  Neutral (0): 2855 (46.1%)
  Emotional (1): 3341 (53.9%)


In [7]:
for subject in sorted(set(groups_en)):
    mask = groups_en == subject
    labels = y_en[mask]
    unique = np.unique(labels)
    print(f"Subject {subject}: {len(labels)} windows, classes: {unique}, counts: {np.bincount(labels)}")

Subject 002: 185 windows, classes: [0 1], counts: [ 18 167]
Subject 003: 400 windows, classes: [0 1], counts: [188 212]
Subject 004: 245 windows, classes: [0 1], counts: [104 141]
Subject 005: 242 windows, classes: [0 1], counts: [155  87]
Subject 007: 511 windows, classes: [0 1], counts: [ 78 433]
Subject 011: 43 windows, classes: [0], counts: [43]
Subject 012: 95 windows, classes: [0 1], counts: [83 12]
Subject 013: 26 windows, classes: [0 1], counts: [12 14]
Subject 014: 56 windows, classes: [0], counts: [56]
Subject 015: 269 windows, classes: [0 1], counts: [122 147]
Subject 016: 213 windows, classes: [0 1], counts: [184  29]
Subject 017: 213 windows, classes: [0 1], counts: [177  36]
Subject 018: 7 windows, classes: [0], counts: [7]
Subject 020: 105 windows, classes: [0 1], counts: [95 10]
Subject 021: 375 windows, classes: [0 1], counts: [162 213]
Subject 022: 207 windows, classes: [0 1], counts: [112  95]
Subject 023: 173 windows, classes: [0 1], counts: [ 73 100]
Subject 024: 3

##### 4.2 Positive vs. Negative (remap + class distribution)
Positive {E4, E5} vs. Negative {E1, E2}

In [8]:
def remap_positive_negative(y):
    y = np.array(y)
    keep_mask = np.isin(y, [1, 2, 4, 5])  
    new_y = np.zeros(len(y), dtype=int)
    new_y[np.isin(y, [4, 5])] = 1  
    return new_y[keep_mask], keep_mask

y_pn, mask_pn = remap_positive_negative(y)
X_pn = X[mask_pn]
groups_pn = np.array(window_groups)[mask_pn]

print(f"\n=== Positive vs. Negative Class Distribution ===")
print(f"  Total: {len(y_pn)}")
print(f"  Negative (0): {np.sum(y_pn == 0)} ({np.sum(y_pn == 0)/len(y_pn)*100:.1f}%)")
print(f"  Positive (1): {np.sum(y_pn == 1)} ({np.sum(y_pn == 1)/len(y_pn)*100:.1f}%)")


=== Positive vs. Negative Class Distribution ===
  Total: 3341
  Negative (0): 1374 (41.1%)
  Positive (1): 1967 (58.9%)


### 5. Model Training
For both models and classification schemes, hyperparameters are tuned once using Optuna with StratifiedGroupKFold (5 splits) on the full dataset, ensuring no subject appears in both train and validation folds. Best parameters are then frozen and used for final LOSO evaluation. Results are also evaluated using 10-fold CV for direct comparison with existing literature.

#### 5.1 XGBoost

##### 5.1.1 Emotional vs. Neutral

In [12]:
def no_improvement_callback(study, trial, n_trials_no_improve=50):
    if trial.number >= n_trials_no_improve:
        recent_values = [t.value for t in study.trials[-n_trials_no_improve:]]
        if max(recent_values) <= study.best_value:
            study.stop()

# Class balancing
neg_count = np.sum(y_en == 0)
pos_count = np.sum(y_en == 1)
scale_en = neg_count / pos_count

# Tune once on full dataset with StratifiedGroupKFold
def objective(trial):
    params = {
        'n_estimators':     trial.suggest_int('n_estimators', 100, 600),
        'max_depth':        trial.suggest_int('max_depth', 3, 8),
        'learning_rate':    trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample':        trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'gamma':            trial.suggest_float('gamma', 0, 5),
        'scale_pos_weight': scale_en,
        'eval_metric':      'logloss',
    }
    model = XGBClassifier(**params, n_jobs=-1)
    cv = StratifiedGroupKFold(n_splits=5)
    scores = cross_val_score(model, X_en, y_en, cv=cv, groups=groups_en, scoring='accuracy', n_jobs=-1)
    return scores.mean()

study = optuna.create_study(direction='maximize')
study.optimize(objective, callbacks=[lambda s, t: no_improvement_callback(s, t)])
print(f"Best params: {study.best_params}")
print(f"Best CV accuracy: {study.best_value:.4f}")

# Save best params
best_params = study.best_params
best_params['scale_pos_weight'] = scale_en
best_params['random_state'] = 42

[I 2026-02-28 16:05:35,940] A new study created in memory with name: no-name-3b43fbc9-25ea-487a-b275-36032636ee49


[I 2026-02-28 16:05:37,933] Trial 0 finished with value: 0.46532439620298727 and parameters: {'n_estimators': 110, 'max_depth': 6, 'learning_rate': 0.06045353619491593, 'subsample': 0.7587638918990197, 'colsample_bytree': 0.6221328843708853, 'min_child_weight': 6, 'gamma': 1.3823102298123202}. Best is trial 0 with value: 0.46532439620298727.
[I 2026-02-28 16:05:39,547] Trial 1 finished with value: 0.5008840034779958 and parameters: {'n_estimators': 288, 'max_depth': 8, 'learning_rate': 0.2376194831365324, 'subsample': 0.6926597952677397, 'colsample_bytree': 0.6251640310008483, 'min_child_weight': 10, 'gamma': 3.1760155404399155}. Best is trial 1 with value: 0.5008840034779958.
[I 2026-02-28 16:05:41,729] Trial 2 finished with value: 0.463034486283739 and parameters: {'n_estimators': 391, 'max_depth': 3, 'learning_rate': 0.01507259765692132, 'subsample': 0.8072377396710296, 'colsample_bytree': 0.8951881213527462, 'min_child_weight': 9, 'gamma': 3.600012106088577}. Best is trial 1 with v

Best params: {'n_estimators': 288, 'max_depth': 8, 'learning_rate': 0.2376194831365324, 'subsample': 0.6926597952677397, 'colsample_bytree': 0.6251640310008483, 'min_child_weight': 10, 'gamma': 3.1760155404399155}
Best CV accuracy: 0.5009


In [13]:
# LOSO evaluation with fixed params
logo = LeaveOneGroupOut()
en_accuracies = []

for fold, (train_idx, test_idx) in enumerate(logo.split(X_en, y_en, groups_en)):
    X_train, X_test = X_en[train_idx], X_en[test_idx]
    y_train, y_test = y_en[train_idx], y_en[test_idx]

    model = XGBClassifier(**best_params, n_jobs=-1)
    model.fit(X_train, y_train)
    preds = model.predict(X_test)

    en_accuracies.append(accuracy_score(y_test, preds))
    print(f"Fold {fold+1} | Accuracy: {en_accuracies[-1]:.4f}")

print("\n=== LOSO XGB (EN) ===")
print(f"Accuracy:  {np.mean(en_accuracies):.4f} ± {np.std(en_accuracies):.4f}")


Fold 1 | Accuracy: 0.6162
Fold 2 | Accuracy: 0.4650
Fold 3 | Accuracy: 0.5224
Fold 4 | Accuracy: 0.5992
Fold 5 | Accuracy: 0.3894
Fold 6 | Accuracy: 0.6744
Fold 7 | Accuracy: 0.7263
Fold 8 | Accuracy: 0.3846
Fold 9 | Accuracy: 0.5893
Fold 10 | Accuracy: 0.3978
Fold 11 | Accuracy: 0.6573
Fold 12 | Accuracy: 0.2911
Fold 13 | Accuracy: 0.5714
Fold 14 | Accuracy: 0.4571
Fold 15 | Accuracy: 0.4960
Fold 16 | Accuracy: 0.5266
Fold 17 | Accuracy: 0.5202
Fold 18 | Accuracy: 0.4628
Fold 19 | Accuracy: 0.4783
Fold 20 | Accuracy: 0.6078
Fold 21 | Accuracy: 0.5124
Fold 22 | Accuracy: 0.5029
Fold 23 | Accuracy: 0.3677
Fold 24 | Accuracy: 0.2055
Fold 25 | Accuracy: 0.4321
Fold 26 | Accuracy: 0.3917
Fold 27 | Accuracy: 0.5136
Fold 28 | Accuracy: 0.4939
Fold 29 | Accuracy: 0.4839
Fold 30 | Accuracy: 0.5397
Fold 31 | Accuracy: 0.5354
Fold 32 | Accuracy: 0.4429
Fold 33 | Accuracy: 0.4779
Fold 34 | Accuracy: 0.4228

=== LOSO XGB (EN) ===
Accuracy:  0.4928 ± 0.1047


In [11]:
# 10 Fold Cross CV with same tuned params
cv_10fold = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
scores_10fold = cross_val_score(
    XGBClassifier(**best_params, n_jobs=-1),
    X_en, y_en,
    cv=cv_10fold,
    scoring='accuracy',
    n_jobs=-1
)

print("\n=== 10-FoldCV XGB (EN) ===")
print(f"Accuracy: {scores_10fold.mean():.4f} ± {scores_10fold.std():.4f}")


=== 10-FoldCV XGB (EN) ===
Accuracy: 0.6791 ± 0.0153
